<a href="https://colab.research.google.com/github/rakasiwisurya/pgaibllm-lessons/blob/main/Latihan_Membangun_AI_Agent_Sederhana_dengan_Framework_ReAct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instalasi Library: Menyiapkan Lingkungan Kerja Agent

In [10]:
!pip install -q -U langchain-groq langchain-community duckduckgo-search pydantic ddgs

In [11]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_community.tools import DuckDuckGoSearchRun
from pydantic import BaseModel, Field
from typing import List

# Konfigurasi: Menghubungkan Agent ke Layanan LLM

In [12]:
# SETUP & KONFIGURASI
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

# Schema Data: Menetapkan Format Output

In [13]:
# DEFINISI SCHEMA (Kontrak Data yang akan dipakai langsung oleh LLM)
class ResearchResult(BaseModel):
    topik: str = Field(description="Topik utama riset")
    prediksi_2026: str = Field(description="Prediksi perkembangan di tahun 2026")
    teknologi_kunci: List[str] = Field(description="Daftar minimal 3 teknologi kunci")
    status_riset: str = Field(description="Status kelengkapan riset (Lengkap/Perlu Tambahan)")

# ReAct Flow: Alur Kerja Agent dari Planning hingga Sintesis

### Thought: Agent Menyusun Rencana

### Action: Agent Mengakses Dunia Luar

### Observation: Agent Mengevaluasi Hasil

### Sintesis: Mengubah Data Menjadi Informasi Terstruktur

In [14]:
def research_agent_react(topic):

  # Inisialisasi LLM & Tool
  llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)
  search = DuckDuckGoSearchRun()



  # --- THOUGHT STEP ---
  # Agent menalar langkah pertama secara mandiri
  reasoning_prompt = f"Tugas: Riset '{topic}'. Berikan satu kalimat pemikiran (thought) tentang informasi spesifik apa yang Anda butuhkan sebelum memberikan jawaban akhir."
  thought = llm.invoke(reasoning_prompt).content
  print(f"1. [THOUGHT]:\n   > {thought}")

  # --- THOUGHT STEP ---
  # Agent menalar langkah pertama secara mandiri
  reasoning_prompt = f"Tugas: Riset '{topic}'. Berikan satu kalimat pemikiran (thought) tentang informasi spesifik apa yang Anda butuhkan sebelum memberikan jawaban akhir."
  thought = llm.invoke(reasoning_prompt).content
  print(f"1. [THOUGHT]:\n   > {thought}")



  # --- ACTION STEP ---
  print(f"\n2. [ACTION]:\n   > Memanggil DuckDuckGo Search API...")
  try:
      observation = search.run(topic)
  except Exception as e:
      observation = f"Gagal mengambil data: {e}"



  # --- OBSERVATION STEP ---
  print(f"\n3. [OBSERVATION]:\n   > Berhasil mengumpulkan {len(observation)} karakter data.")



  # --- STRUCTURED SYNTHESIS STEP ---
  # Menggunakan with_structured_output untuk menjamin JSON valid sesuai class ResearchResult
  print(f"\n4. [FINAL THOUGHT]:\n   > Melakukan validasi dan pemformatan data ke JSON...")

  structured_llm = llm.with_structured_output(ResearchResult)
  final_prompt = f"Berdasarkan data berikut, buatlah laporan riset terstruktur: {observation}"

  # Hasil akhir langsung berupa objek ResearchResult, bukan string teks biasa
  result = structured_llm.invoke(final_prompt)
  return result

# Eksekusi: Menjalankan Agent dalam Satu Alur

In [15]:
# EKSEKUSI
if __name__ == "__main__":
    topik_target = "Masa depan Agentic AI di tahun 2026"
    hasil = research_agent_react(topik_target)

    print("\n" + "="*40)
    print("HASIL AKHIR AGENT (VALIDATED OBJECT)")
    print("="*40)
    print(f"Topik          : {hasil.topik}")
    print(f"Prediksi 2026  : {hasil.prediksi_2026}")
    print(f"Teknologi Kunci: {', '.join(hasil.teknologi_kunci)}")
    print(f"Status         : {hasil.status_riset}")

1. [THOUGHT]:
   > Sebelum memberikan jawaban akhir tentang masa depan Agentic AI di tahun 2026, saya memerlukan informasi spesifik tentang perkembangan terbaru dalam bidang kecerdasan buatan, terutama terkait kemajuan dalam pengembangan agen pintar yang dapat berinteraksi dan membuat keputusan secara otonom.
1. [THOUGHT]:
   > Sebelum memberikan jawaban akhir tentang masa depan Agentic AI di tahun 2026, saya memerlukan informasi spesifik tentang perkembangan terbaru dalam bidang kecerdasan buatan, terutama terkait kemajuan dalam pengembangan agen pintar yang dapat berinteraksi dan beradaptasi dengan lingkungan mereka secara mandiri.

2. [ACTION]:
   > Memanggil DuckDuckGo Search API...

3. [OBSERVATION]:
   > Berhasil mengumpulkan 1151 karakter data.

4. [FINAL THOUGHT]:
   > Melakukan validasi dan pemformatan data ke JSON...

HASIL AKHIR AGENT (VALIDATED OBJECT)
Topik          : Agentic AI
Prediksi 2026  : Tahun 2026 menandai tahun perusahaan melintasi kesenjangan dari pilot ke produ